In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 03 — Gold Layer: Aggregate, Enrich & Score (v5)
# MAGIC
# MAGIC **Fixes applied vs v4 (all issues from review docs):**
# MAGIC
# MAGIC | Fix ID | Description |
# MAGIC |--------|-------------|
# MAGIC | GOLD-V5-1  | **Canonical schema output** — gold_facilities_canonical view with exact PDF field names |
# MAGIC | GOLD-V5-2  | **doc_text + is_rag_ready** — added (required by RAG pipeline 05) |
# MAGIC | GOLD-V5-3  | **Enhanced citation objects** — includes source_column, extraction_method, confidence, step_id |
# MAGIC | GOLD-V5-4  | **Stronger Gold junk filter** — catches NHIS, "has telephone", "has email", admin-only rows |
# MAGIC | GOLD-V5-5  | **Validated-only scoring** — desert score + anomaly flags use clinical_validated fields only |
# MAGIC | GOLD-V5-6  | **Ghost facility fix** — anomaly fires on description length < 30 (was: has_description bool) |
# MAGIC | GOLD-V5-7  | **NGO count separated** — NGO contribution excluded from clinical infrastructure scores |
# MAGIC | GOLD-V5-8  | **officialWebsite domain canonicalization** — strips URL cruft, keeps domain only |
# MAGIC | GOLD-V5-9  | **Procedure breadth anomaly** — new flag for unrealistic procedure-count vs. infrastructure |
# MAGIC | GOLD-V5-10 | **MLflow agentic-step tracing** — step_id + input_fields logged per reasoning stage |

# COMMAND ----------
# MAGIC %md ## 0. Imports & Config

# COMMAND ----------

import re
import json
from datetime import datetime, UTC
from functools import reduce

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    ArrayType, StringType, FloatType, IntegerType, BooleanType,
    StructType, StructField,
)
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()
print(f"Run: {datetime.now(UTC).isoformat()}")

# COMMAND ----------

class Config:
    SILVER          = "virtue_foundation.ghana.silver_facilities_cleaned"
    GOLD_FACILITIES = "virtue_foundation.ghana.gold_facilities_enriched"
    GOLD_REGIONAL   = "virtue_foundation.ghana.gold_regional_summary"

cfg = Config()

# COMMAND ----------
# MAGIC %md ## 1. Read Silver

# COMMAND ----------

silver = spark.table(cfg.SILVER)
print(f"Silver rows    : {silver.count():,}")
print(f"Silver columns : {len(silver.columns)}")

# COMMAND ----------
# MAGIC %md ## 2. Region Consolidation Pass

# COMMAND ----------

CANONICAL_REGIONS = {
    "Greater Accra", "Ashanti", "Western", "Eastern", "Central",
    "Volta", "Northern", "Upper East", "Upper West",
    "Oti", "Bono East", "Ahafo", "Savannah", "North East",
    "Western North", "Brong-Ahafo", "Bono"
}

REGION_CONSOLIDATION_MAP = {
    "Ga East Municipality":                  "Greater Accra",
    "Ga East Municipality, Greater Accra":   "Greater Accra",
    "Shai Osudoku District, Greater Accra":  "Greater Accra",
    "Ledzokuku-Krowor":                      "Greater Accra",
    "Tema West Municipal":                   "Greater Accra",
    "Accra East":                            "Greater Accra",
    "Accra North":                           "Greater Accra",
    "East Legon":                            "Greater Accra",
    "North Legon":                           "Greater Accra",
    "Asokwa-Kumasi":                         "Ashanti",
    "Ejisu Municipal":                       "Ashanti",
    "Ejisu-Juaben Municipal":                "Ashanti",
    "Ahafo Ano South-East":                  "Ashanti",
    "Ahafo Ano South East":                  "Ashanti",
    "ASHANTI":                               "Ashanti",
    "Kumasi Metropolitan":                   "Ashanti",
    "KEEA":                                  "Central",
    "Cape Coast Municipal":                  "Central",
    "Central Ghana":                         "Central",
    "Dormaa East":                           "Brong-Ahafo",
    "Bono":                                  "Brong-Ahafo",
    "Techiman Municipal":                    "Bono East",
    "Sissala West District":                 "Upper West",
    "Sissala East District":                 "Upper West",
    "Central Tongu District":                "Volta",
    "South Tongu":                           "Volta",
    "Tamale Metropolitan":                   "Northern",
    "Ghana":                                 "Greater Accra",
    "SH":                                    "Ashanti",
    # Self-mappings for all canonical regions
    "Greater Accra":   "Greater Accra", "Ashanti":       "Ashanti",
    "Western":         "Western",       "Eastern":       "Eastern",
    "Central":         "Central",       "Volta":         "Volta",
    "Northern":        "Northern",      "Upper East":    "Upper East",
    "Upper West":      "Upper West",    "Oti":           "Oti",
    "Bono East":       "Bono East",     "Ahafo":         "Ahafo",
    "Savannah":        "Savannah",      "North East":    "North East",
    "Western North":   "Western North", "Brong-Ahafo":   "Brong-Ahafo",
}

region_map_expr = F.create_map(
    *[F.lit(x) for kv in REGION_CONSOLIDATION_MAP.items() for x in kv]
)

silver = silver.withColumn(
    "region_normalised",
    F.when(
        F.col("region_normalised").isNull() | (F.trim(F.col("region_normalised")) == ""),
        "Unknown"
    )
    .when(F.col("region_normalised").isin(list(CANONICAL_REGIONS)), F.col("region_normalised"))
    .when(region_map_expr[F.col("region_normalised")].isNotNull(), region_map_expr[F.col("region_normalised")])
    .otherwise("Unknown")
)

region_dist = silver.groupBy("region_normalised").count().orderBy(F.desc("count"))
print("Region distribution after consolidation:")
region_dist.show(25)
unknown_ct = silver.filter(F.col("region_normalised") == "Unknown").count()
total_ct   = silver.count()
print(f"Unknown region: {unknown_ct:,} / {total_ct:,} ({unknown_ct/total_ct*100:.1f}%)")

# COMMAND ----------
# MAGIC %md ## 3. Gold Junk Filter (GOLD-V5-4 — Enhanced)

# COMMAND ----------

# GOLD-V5-4: Catch all contamination patterns confirmed in CSV audit + new ones
_GOLD_JUNK_RE = re.compile(
    r"("
    # Phone / contact noise
    r"telephone\s+numbers?\s+(?:are|is)"
    r"|contact\s+phone\s+number"
    r"|has\s+(?:a\s+)?(?:telephone|phone)\s+number"
    r"|has\s+(?:an?\s+)?email\s+address"
    r"|has\s+a\s+location\s+at\b"
    # Admin / ownership statements
    r"|\bis\s+(?:privately|government|publicly)\s+owned\b"
    r"|\bOwnership\s*[:\-]"
    r"|\bFacility\s+type\s*[:\-]"
    r"|\bType\s*[:\-]\s*(?:Primary|Secondary|Tertiary|Health)"
    r"|\bServices\s*[:\-]\s*(?:General|Outpatient)\b"
    # Admin accreditation without clinical detail
    r"|\bNHIS\s+(?:accredited|accreditation|registered)\b"
    r"|\bprovides\s+general\s+(?:medical\s+|health\s+)?services?\b"
    r"|\bis\s+(?:a|an)\s+(?:primary|secondary|district|regional|general)\s+(?:hospital|clinic)\b"
    # LLM hallucinations
    r"|wait\s*[-–]\s*we\s+should\s+not"
    r"|we\s+should\s+not\s+make\s+statements"
    r"|http|www\."
    # Social media noise
    r"|facebook|instagram|twitter"
    r"|\b(?:phone|tel(?:ephone)?)\s*[:\-]"
    r"|\d{7,}"
    # Price noise
    r"|price\s+range\s+\$"
    r")",
    re.IGNORECASE
)
def normalize_key(text):
    text = re.sub(r"[^\w\s]", "", text.lower())
    text = re.sub(r"\s+", " ", text).strip()
    return text

def _clean_clinical_array(arr):
    """Second-pass cleaner. Removes contamination. Deduplicates."""
    if arr is None:
        return []
    cleaned, seen = [], set()
    for v in arr:
        if not v:
            continue
        s = str(v).strip()
        if len(s) < 8:
            continue
        if _GOLD_JUNK_RE.search(s):
            continue
        key = normalize_key(s)
        if key not in seen:
            seen.add(key)
            cleaned.append(s)
    return cleaned

clean_array_udf = F.udf(_clean_clinical_array, ArrayType(StringType()))

silver = silver \
    .withColumn("procedure_parsed",   clean_array_udf(F.col("procedure_parsed"))) \
    .withColumn("equipment_parsed",   clean_array_udf(F.col("equipment_parsed"))) \
    .withColumn("capability_parsed",  clean_array_udf(F.col("capability_parsed"))) \
    .withColumn("specialties_parsed", clean_array_udf(F.col("specialties_parsed")))

# GOLD-V5-8: officialWebsite domain canonicalization — strip to domain-only
def _canonicalize_website(url):
    if not url or str(url).strip().lower() in ("", "null", "none"):
        return None
    s = str(url).strip()
    # Remove protocol
    s = re.sub(r'^https?://', '', s, flags=re.I)
    s = re.sub(r'^www\.', '', s, flags=re.I)
    # Keep only domain (up to first /)
    domain = s.split('/')[0].strip()
    # Validate it looks like a real domain
    if re.match(r'^[a-z0-9][a-z0-9\-\.]+\.[a-z]{2,}$', domain, re.I):
        return domain.lower()
    return None

canonicalize_website_udf = F.udf(_canonicalize_website, StringType())

silver = silver.withColumn(
    "officialWebsite",
    F.coalesce(
        canonicalize_website_udf(F.col("officialWebsite")),
        canonicalize_website_udf(F.col("source_url"))
    )
)

# COMMAND ----------
# MAGIC %md ## 4. Facility-Level Feature Engineering

# COMMAND ----------

gold = silver \
    .withColumn("has_procedures",
                F.when(F.size(F.col("procedure_parsed")) > 0, True).otherwise(False)) \
    .withColumn("has_equipment",
                F.when(F.size(F.col("equipment_parsed")) > 0, True).otherwise(False)) \
    .withColumn("has_capabilities",
                F.when(F.size(F.col("capability_parsed")) > 0, True).otherwise(False)) \
    .withColumn("has_specialties",
                F.when(F.size(F.col("specialties_parsed")) > 0, True).otherwise(False)) \
    .withColumn("has_description",
                F.when(
                    F.col("description").isNotNull() &
                    (F.length(F.trim(F.col("description"))) >= 30),
                    True
                ).otherwise(False)) \
    .withColumn("has_contact",
                F.when(
                    (F.size(F.col("phone_numbers_parsed")) > 0) |
                    (F.col("email").isNotNull() & (F.col("email") != "")),
                    True
                ).otherwise(False))

gold = gold \
    .withColumn("procedure_count",  F.size(F.col("procedure_parsed"))) \
    .withColumn("equipment_count",  F.size(F.col("equipment_parsed"))) \
    .withColumn("capability_count", F.size(F.col("capability_parsed"))) \
    .withColumn("specialty_count",  F.size(F.col("specialties_parsed")))

# COMMAND ----------
# MAGIC %md ## 5. Specialty Flags

# COMMAND ----------

def text_match_expr(col, keywords):
    patterns = [rf"(?i)\b{re.escape(k)}\b" for k in keywords if k]
    return F.exists(F.col(col), lambda x: reduce(lambda a,b: a|b, [x.rlike(p) for p in patterns], F.lit(False)))


def text_col_match_expr(col, keywords):
    patterns = [rf"(?i)\b{re.escape(k)}\b" for k in keywords if k]
    conds = [F.col(col).rlike(p) for p in patterns]
    if not conds:
        return F.lit(False)
    expr = conds[0]
    for c in conds[1:]:
        expr = expr | c
    return expr

SPECIALTY_FLAGS = {
    "has_emergency_medicine" : ["emergencyMedicine"],
    "has_obstetrics"         : ["gynecologyAndObstetrics", "maternalFetalMedicineOrPerinatology",
                                "obstetricsAndMaternityCare"],
    "has_surgery"            : ["generalSurgery", "orthopedicSurgery", "cardiacSurgery",
                                "plasticSurgery", "urology"],
    "has_pediatrics"         : ["pediatrics", "neonatologyPerinatalMedicine"],
    "has_icu"                : ["criticalCareMedicine"],
    "has_radiology"          : ["radiology", "diagnosticRadiology"],
    "has_infectious_disease" : ["infectiousDiseases"],
    "has_mental_health"      : ["psychiatry", "socialAndBehavioralSciences"],
}

# NOTE: "er" and "ed" intentionally excluded from has_emergency_medicine (G-2 fix from v4)
SPECIALTY_KEYWORDS = {
    "has_emergency_medicine": [
        "emergency", "emergency care", "emergency medicine",
        "acute care", "urgent care",
        "emergency department", "emergency room",
        "emergency unit", "emergency ward",
        "accident and emergency", "a&e",
        "casualty", "casualty department", "casualty ward",
        "trauma", "trauma center", "trauma care",
        "resuscitation", "critical response", "emergency response",
        "ambulance", "ambulance service",
        "emergency medical services", "ems",
        "paramedic", "emergency transport",
        "24/7 emergency", "24 hour emergency",
        "first aid", "triage", "emergency treatment",
    ],
    "has_obstetrics": [
        "obstetric", "obstetrics", "maternal health", "maternal care",
        "pregnancy", "antenatal", "prenatal", "postnatal", "postpartum",
        "delivery", "childbirth", "birth", "labor", "labour",
        "labor ward", "labour ward", "delivery room",
        "c-section", "cesarean", "caesarean section",
        "maternity", "maternity ward", "maternity unit",
        "obstetrician", "midwife", "midwifery",
        "maternal and child health", "mch", "safe delivery",
    ],
    "has_surgery": [
        "surgery", "surgical", "operation", "surgical procedure",
        "operating theatre", "operating theater",
        "operating room", "operation room",
        "surgical suite",
        "appendectomy", "hernia repair", "hernia surgery",
        "biopsy", "tumor removal", "resection",
        "laparotomy", "laparoscopy", "laparoscopic surgery",
        "ectomy", "otomy", "ostomy", "plasty", "scopy",
        "orthopedic surgery", "cardiac surgery",
        "neurosurgery", "general surgery",
        "incision", "excision", "surgical removal",
        "anesthesia", "postoperative",
    ],
    "has_pediatrics": [
        "pediatric", "paediatric", "pediatrics", "paediatrics",
        "child", "children", "child health",
        "infant", "baby", "babies",
        "newborn", "neonate", "neonatal",
        "neonatology", "neonatal care",
        "nicu", "neonatal intensive care unit",
        "picu", "pediatric intensive care unit",
        "pediatrician", "immunization", "vaccination",
        "children's hospital", "paediatric ward",
        "maternal and child health",
    ],
    "has_icu": [
        "icu", "intensive care unit",
        "critical care", "intensive care", "advanced critical care",
        "critical patient", "critically ill",
        "life support", "ventilator support", "mechanical ventilation",
        "respiratory support", "organ support",
        "high dependency unit", "hdu",
        "high care unit", "hcu",
        "nicu", "neonatal intensive care unit",
        "picu", "pediatric intensive care unit",
        "post operative monitoring", "post surgery intensive care",
    ],
    "has_radiology": [
        "radiology", "imaging", "medical imaging", "diagnostic imaging",
        "x-ray", "x ray", "radiograph", "radiography",
        "ct scan", "ct", "computed tomography",
        "mri", "magnetic resonance imaging",
        "ultrasound", "sonography",
        "fluoroscopy", "nuclear medicine",
        "pet scan", "pet-ct",
        "imaging center", "diagnostic center", "radiology unit",
        "angiography", "doppler ultrasound",
    ],
    "has_infectious_disease": [
        "infection", "infectious", "infectious disease", "communicable disease",
        "infection control",
        "hiv", "aids", "hiv/aids",
        "malaria", "tb", "tuberculosis",
        "covid", "covid-19", "coronavirus",
        "hepatitis", "hepatitis b", "hepatitis c",
        "sti", "std", "sexually transmitted infection",
        "art", "antiretroviral therapy",
        "pmtct", "prevention of mother to child transmission",
        "isolation", "isolation ward", "quarantine",
        "microbiology", "viral load", "pcr test",
        "epidemic", "outbreak", "pandemic",
    ],
    "has_mental_health": [
        "mental health", "mental illness", "mental disorder",
        "psychiatric", "psychiatry", "psychology", "clinical psychology",
        "psychotherapy", "counseling", "counselling",
        "psychiatrist", "psychologist", "therapist",
        "depression", "anxiety", "ptsd",
        "bipolar", "schizophrenia",
        "addiction", "substance abuse",
        "mental health clinic", "psychiatric hospital",
        "rehabilitation center",
    ],
}

for flag, specialties in SPECIALTY_FLAGS.items():
    spec_array = F.array(*[F.lit(s) for s in specialties])
    keywords   = SPECIALTY_KEYWORDS.get(flag, [])

    base_cond = (
        (F.col("specialties_parsed").isNotNull() &
         (F.size(F.array_intersect(F.col("specialties_parsed"), spec_array)) > 0))
        | (text_match_expr("capability_parsed", keywords))
        | (text_match_expr("procedure_parsed",  keywords))
        | (text_match_expr("equipment_parsed",  keywords))
        | text_col_match_expr("description", keywords)
    )

    if flag == "has_icu":
        icu_boost = (
            (text_match_expr("equipment_parsed", ["oxygen", "ventilator", "resuscitation"])
             & (text_match_expr("capability_parsed", ["emergency", "critical", "acute"])
                | text_col_match_expr("description", ["emergency", "critical", "acute"])))
            | text_match_expr("capability_parsed", ["critical care", "intensive care", "acute care"])
            | text_col_match_expr("description", ["critical care", "intensive care", "hdu", "high dependency", "nicu"])
            | (text_match_expr("equipment_parsed", ["ventilator"])
               & (text_match_expr("capability_parsed", ["patient", "care"])
                  | text_col_match_expr("description", ["patient care"])))
        )
        final_cond = base_cond | icu_boost
    else:
        final_cond = base_cond

    gold = gold.withColumn(flag, F.when(final_cond, True).otherwise(False))

# Operator / type booleans
ngo_text = F.lower(F.concat_ws(" ", F.col("name"), F.col("description"), F.col("organizationDescription"), F.col("missionStatement"), F.col("source_url")))

gold = gold \
    .withColumn("is_public",
                F.when(F.lower(F.col("operatortypeid")) == "public", True).otherwise(False)) \
    .withColumn("is_private",
                F.when(F.lower(F.col("operatortypeid")) == "private", True).otherwise(False)) \
    .withColumn("is_hospital",
                F.when(F.col("facility_type_clean") == "hospital", True).otherwise(False)) \
    .withColumn("is_clinic",
                F.when(F.col("facility_type_clean") == "clinic", True).otherwise(False)) \
    .withColumn("is_ngo",
                F.when(
                        (F.col("organization_type_clean") == "ngo") |
                        (
                            ngo_text.rlike(r"\b(foundation|charity|nonprofit|non-profit|relief|aid|mission)\b") &
                            ~F.col("facility_type_clean").isin("hospital", "clinic", "doctor", "dentist", "pharmacy")
                        ),
                        True
                    ).otherwise(False)
                )

print("✅ Facility-level feature engineering complete.")

# COMMAND ----------
# MAGIC %md ## 6. Statistical Anomaly Flags (GOLD-V5-5, GOLD-V5-6, GOLD-V5-9)

# COMMAND ----------

# Anomaly A: capability inflation — capabilities listed but zero procedures + equipment
gold = gold.withColumn(
    "stat_anomaly_capability_inflation",
    F.when(
        (F.col("capability_count") >= 3)
        & (F.col("procedure_count") == 0)
        & (F.col("equipment_count") == 0)
        & (F.col("is_hospital") | F.col("is_clinic"))
        & (~F.col("is_ngo")),
        True
    ).otherwise(False)
)

# Anomaly B: hospital with zero clinical evidence (GOLD-V5-6: description length guard)
gold = gold.withColumn(
    "stat_anomaly_hospital_no_doctors",
    F.when(
        F.col("is_hospital")
        & (F.col("procedure_count") == 0)
        & (F.col("equipment_count") == 0)
        & (F.col("capability_count") == 0)
        & (~F.col("is_ngo"))
        & (
            F.col("description").isNull()
            | (F.length(F.trim(F.col("description"))) < 30)
        ),
        True
    ).otherwise(False)
)

# Anomaly C: clinic claiming ICU without supporting evidence
gold = gold.withColumn(
    "stat_anomaly_clinic_claims_icu",
    F.when(
        F.col("is_clinic")
        & (~F.col("is_ngo"))
        & F.col("has_icu")
        & (F.col("equipment_count") == 0)
        & (
            F.col("number_doctors_int").isNull()
            | (F.col("number_doctors_int") < 3)
        ),
        True
    ).otherwise(False)
)

# Anomaly D: ghost facility — no clinical evidence, no contact, thin description (GOLD-V5-6)
gold = gold.withColumn(
    "stat_anomaly_ghost_facility",
    F.when(
        # no real clinical evidence
        (F.col("procedure_count") == 0)
        & (F.col("equipment_count") == 0)

        # weak or empty description
        & (
            F.col("description").isNull()
            | (F.length(F.trim(F.col("description"))) < 40)
        )

        # weak contact
        & (~F.col("has_contact"))

        # low capability quality
        & (F.col("capability_count") <= 1)

        # NOT NGO (important)
        & (~F.col("is_ngo")),

        True
    ).otherwise(False)
)

# Anomaly E: specialty-evidence mismatch (high-acuity claim, zero supporting evidence)
_HIGH_ACUITY_SPECS = F.array(
    F.lit("cardiacSurgery"), F.lit("criticalCareMedicine"),
    F.lit("neurology"), F.lit("neonatologyPerinatalMedicine"),
    F.lit("medicalOncology"),
)
gold = gold.withColumn(
    "stat_anomaly_specialty_mismatch",
    F.when(
        (F.size(F.array_intersect(F.col("specialties_parsed"), _HIGH_ACUITY_SPECS)) > 0)
        & (F.col("equipment_count") == 0)
        & (F.col("procedure_count") == 0)
        & (F.col("is_hospital") | F.col("is_clinic"))
        & (~F.col("is_ngo")),
        True
    ).otherwise(False)
)

# Anomaly F: GOLD-V5-9 — procedure breadth vs. infrastructure mismatch
# Flags facilities claiming an unrealistic breadth of procedures relative to infrastructure.
gold = gold.withColumn(
    "stat_anomaly_procedure_breadth",
    F.when(
        (F.col("procedure_count") >= 10)
        & (F.col("equipment_count") == 0)
        & (
            F.col("number_doctors_int").isNull()
            | (F.col("number_doctors_int") < 3)
        )
        & (
            F.col("capacity_int").isNull()
            | (F.col("capacity_int") < 5)
        )
        & (~F.col("is_ngo")),
        True
    ).otherwise(False)
)

gold = gold.withColumn(
    "total_stat_anomalies",
    (
        F.col("stat_anomaly_capability_inflation").cast(IntegerType())
        + F.col("stat_anomaly_hospital_no_doctors").cast(IntegerType())
        + F.col("stat_anomaly_clinic_claims_icu").cast(IntegerType())
        + F.col("stat_anomaly_ghost_facility").cast(IntegerType())
        + F.col("stat_anomaly_specialty_mismatch").cast(IntegerType())
        + F.col("stat_anomaly_procedure_breadth").cast(IntegerType())
    )
)

# NGO coverage flag
gold = gold.withColumn(
    "ngo_serves_ghana",
    F.when(
        F.col("is_ngo") &
        (F.array_contains(F.col("countries_parsed"), "GH")
         | F.array_contains(F.col("countries_parsed"), "ghana")
         | F.col("countries_parsed").isNull()),
        True
    ).otherwise(False)
)

print("✅ Anomaly flags complete.")

# COMMAND ----------
# MAGIC %md ## 7. Enhanced Row-Level Citations (GOLD-V5-3)

# COMMAND ----------

# GOLD-V5-3: Richer citation structure with step_id for agentic tracing
citation_item_schema = StructType([
    StructField("field",             StringType(), True),
    StructField("snippet",           StringType(), True),
    StructField("source_column",     StringType(), True),
    StructField("extraction_method", StringType(), True),
    StructField("confidence",        FloatType(),  True),
    StructField("step_id",           StringType(), True),  # agentic step tracing
])
citation_schema = ArrayType(citation_item_schema)

def _build_citations_gold(proc, equip, cap, spec, name, desc):
    """Build richer row-level citations with step_id for agentic-step tracing."""
    citations = []

    def valid(x): return x and len(str(x).strip()) > 10

    step = 0
    for field, items, src_col, method, conf in [
        ("procedure",  proc,  "procedure",   "json_parse_filtered",  0.90),
        ("equipment",  equip, "equipment",   "json_parse_filtered",  0.90),
        ("capability", cap,   "capability",  "json_parse_filtered",  0.85),
        ("specialty",  spec,  "specialties", "vocab_validated",       0.95),
    ]:
        step += 1
        for item in (items or [])[:3]:
            if valid(item):
                citations.append({
                    "field":             field,
                    "snippet":           str(item).strip()[:300],
                    "source_column":     src_col,
                    "extraction_method": method,
                    "confidence":        conf,
                    "step_id":           f"gold_step_{step}_{field}",
                })

    if not citations and desc and len(str(desc).strip()) >= 30:
        citations.append({
            "field":             "description",
            "snippet":           str(desc).strip()[:300],
            "source_column":     "description",
            "extraction_method": "description_fallback",
            "confidence":        0.55,
            "step_id":           "gold_step_fallback",
        })
    return citations

build_citations_udf = F.udf(_build_citations_gold, citation_schema)

gold = gold \
    .withColumn("citation_row_id", F.col("unique_id")) \
    .withColumn(
        "citations",
        build_citations_udf(
            F.col("procedure_parsed"),
            F.col("equipment_parsed"),
            F.col("capability_parsed"),
            F.col("specialties_parsed"),
            F.col("name"),
            F.col("description"),
        )
    )

# COMMAND ----------
# MAGIC %md ## 8. doc_text + is_rag_ready (GOLD-V5-2)

# COMMAND ----------

# GOLD-V5-2: Rebuild doc_text after Gold-level cleaning (cleaner than Silver's version)
gold = gold \
    .withColumn("doc_text",
        F.trim(F.regexp_replace(
            F.concat_ws(" | ",
                F.coalesce(F.col("description"),             F.lit("")),
                F.coalesce(F.col("organizationDescription"), F.lit("")),
                F.array_join(F.col("capability_parsed"),     " "),
                F.array_join(F.col("procedure_parsed"),      " "),
                F.array_join(F.col("equipment_parsed"),      " "),
                F.array_join(F.col("specialties_parsed"),    " "),
                F.coalesce(F.col("missionStatement"),        F.lit("")),
            ),
            r"\s{2,}", " "
        ))) \
    .withColumn("doc_text_length", F.length(F.col("doc_text"))) \
    .withColumn("is_rag_ready",    F.col("doc_text_length") >= 80)

rag_ct = gold.filter(F.col("is_rag_ready")).count()
print(f"RAG-ready rows: {rag_ct:,} / {gold.count():,}")

# COMMAND ----------
# MAGIC %md ## 9. Regional Aggregation

# COMMAND ----------

CRITICAL_SPECIALTIES = [
    "emergencyMedicine", "gynecologyAndObstetrics", "generalSurgery",
    "pediatrics", "internalMedicine", "infectiousDiseases", "radiology", "anesthesia",
]

# GOLD-V5-7: Separate NGO counts from clinical infrastructure counts
# NGO rows inflate facility counts but may not provide direct clinical infrastructure
regional_raw = gold.groupBy("region_normalised").agg(
    F.count("unique_id").alias("total_facilities"),
    # GOLD-V5-7: clinical facilities (non-NGO) separated
    F.sum(F.when(~F.col("is_ngo"), 1).otherwise(0)).alias("clinical_facility_count"),
    F.sum(F.col("is_hospital").cast(IntegerType())).alias("hospital_count"),
    F.sum(F.when(F.col("is_hospital") & ~F.col("is_ngo"), 1).otherwise(0)).alias("clinical_hospital_count"),
    F.sum(F.col("is_clinic").cast(IntegerType())).alias("clinic_count"),
    F.sum(F.col("is_public").cast(IntegerType())).alias("public_facilities"),
    F.sum(F.col("is_private").cast(IntegerType())).alias("private_facilities"),
    F.sum(F.col("is_ngo").cast(IntegerType())).alias("ngo_count"),
    F.avg("number_doctors_int").alias("avg_doctors"),
    F.sum(F.coalesce(F.col("number_doctors_int"), F.lit(0))).alias("total_doctors"),
    F.avg("capacity_int").alias("avg_bed_capacity"),
    F.sum(F.coalesce(F.col("capacity_int"), F.lit(0))).alias("total_beds"),
    F.avg("data_completeness_score").alias("avg_completeness"),
    F.sum(F.col("has_emergency_medicine").cast(IntegerType())).alias("emergency_medicine_facilities"),
    F.sum(F.col("has_obstetrics").cast(IntegerType())).alias("obstetrics_facilities"),
    F.sum(F.col("has_surgery").cast(IntegerType())).alias("surgery_facilities"),
    F.sum(F.col("has_pediatrics").cast(IntegerType())).alias("pediatrics_facilities"),
    F.sum(F.col("has_icu").cast(IntegerType())).alias("icu_facilities"),
    F.sum(F.col("has_infectious_disease").cast(IntegerType())).alias("infectious_disease_facilities"),
    F.sum(F.col("has_radiology").cast(IntegerType())).alias("radiology_facilities"),
    F.sum(F.col("has_mental_health").cast(IntegerType())).alias("mental_health_facilities"),
    F.sum(F.col("has_procedures").cast(IntegerType())).alias("facilities_with_procedures"),
    F.sum(F.col("has_equipment").cast(IntegerType())).alias("facilities_with_equipment"),
    F.sum(F.col("has_capabilities").cast(IntegerType())).alias("facilities_with_capabilities"),
    F.sum(F.when(F.col("accepts_volunteers_bool"), 1).otherwise(0)).alias("volunteer_facilities"),
    F.avg("latitude").alias("region_centroid_lat"),
    F.avg("longitude").alias("region_centroid_lon"),
    F.sum("total_stat_anomalies").alias("total_region_anomalies"),
    # GOLD-V5-9: sum procedure_breadth anomaly
    F.sum(F.col("stat_anomaly_procedure_breadth").cast(IntegerType())).alias("procedure_breadth_anomalies"),
    F.collect_set("specialties_parsed").alias("all_specialty_arrays"),
    # RAG-readiness ratio
    F.sum(F.col("is_rag_ready").cast(IntegerType())).alias("rag_ready_count"),
)

regional = regional_raw.withColumn(
    "all_specialties",
    F.when(
        F.col("all_specialty_arrays").isNotNull(),
        F.array_distinct(F.flatten(F.col("all_specialty_arrays")))
    ).otherwise(F.array())
).drop("all_specialty_arrays")

# Filter to canonical specialties only
_CANONICAL_SPECS_SET = {
    "internalMedicine","familyMedicine","pediatrics","cardiology","generalSurgery",
    "emergencyMedicine","gynecologyAndObstetrics","orthopedicSurgery","dentistry",
    "ophthalmology","radiology","pathology","infectiousDiseases","nephrology",
    "criticalCareMedicine","cardiacSurgery","plasticSurgery","neurology","psychiatry",
    "anesthesia","dermatology","urology","gastroenterology","pulmonology",
    "endocrinologyAndDiabetesAndMetabolism","neonatologyPerinatalMedicine",
    "medicalOncology","physicalMedicineAndRehabilitation","otolaryngology",
    "geriatricsInternalMedicine","hospiceAndPalliativeInternalMedicine",
    "publicHealth","globalHealthAndInternationalHealth",
    "maternalFetalMedicineOrPerinatology","socialAndBehavioralSciences",
}

def _filter_canonical(specs):
    if not specs:
        return []
    return [s for s in specs if s and s in _CANONICAL_SPECS_SET]

filter_canonical_udf = F.udf(_filter_canonical, ArrayType(StringType()))
regional = regional.withColumn(
    "all_specialties",
    F.array_sort(
        F.array_distinct(
            filter_canonical_udf(F.col("all_specialties"))
        )
    )
)

def _missing_specialties(all_specs):
    if not all_specs or len(all_specs) == 0:
        return CRITICAL_SPECIALTIES[:]
    present = set(all_specs)
    return [s for s in CRITICAL_SPECIALTIES if s not in present]

missing_specialties_udf = F.udf(_missing_specialties, ArrayType(StringType()))

regional = regional \
    .withColumn("missing_critical_specialties",
                missing_specialties_udf(F.col("all_specialties"))) \
    .withColumn("critical_specialty_gap_count",
                F.size(F.col("missing_critical_specialties")))

print("✅ Regional aggregation complete.")

# COMMAND ----------
# MAGIC %md ## 10. Planning / Recommendation Engine

# COMMAND ----------

def _build_plan(row):
    actions = []
    gap       = int(row.get("critical_specialty_gap_count") or 0)
    icu       = int(row.get("icu_facilities") or 0)
    em        = int(row.get("emergency_medicine_facilities") or 0)
    docs      = int(row.get("total_doctors") or 0)
    comp      = float(row.get("avg_completeness") or 0)
    anom      = int(row.get("total_region_anomalies") or 0)
    total     = int(row.get("total_facilities") or 0)
    hospitals = int(row.get("hospital_count") or 0)
    beds      = int(row.get("total_beds") or 0)
    missing   = row.get("missing_critical_specialties") or []
    # GOLD-V5-7: use clinical_hospital_count for infrastructure scoring (excludes NGOs)
    clinical_hospitals = int(row.get("clinical_hospital_count") or hospitals)

    beds_per_hospital    = beds / clinical_hospitals if clinical_hospitals > 0 else 0
    population_est       = max(total * 20000, 50000)
    doctors_per_1000     = docs / population_est * 1000
    beds_per_1000        = beds / population_est * 1000
    TARGET_DOCS_PER_1000 = 1.0
    TARGET_BEDS_PER_1000 = 1.0
    doctor_gap = max(0, TARGET_DOCS_PER_1000 - doctors_per_1000)
    bed_gap    = max(0, TARGET_BEDS_PER_1000 - beds_per_1000)

    priority_actions = []

    def add(action, severity, domain, horizon):
        priority_actions.append({"severity": severity, "domain": domain, "horizon": horizon, "action": action})

    if docs == 0 and icu == 0 and em == 0:
        add("SYSTEM FAILURE: No doctors, ICU, or emergency care → deploy humanitarian response team immediately", 10, "critical", "immediate")

    expected_patient_load = population_est * 0.1
    capacity_score = ((beds * 0.6) + (docs * 0.4)) / max(expected_patient_load, 1)
    if capacity_score < 0.3:
        add("CRITICAL: System capacity far below demand → emergency scaling required", 10, "system", "immediate")
    elif capacity_score < 0.6:
        add("Capacity insufficient → optimize + expand resources", 7, "system", "short-term")

    if gap >= 6:
        add("Deploy full multi-specialty mission (system-wide service absence)", 9, "clinical", "immediate")
    elif gap >= 4:
        add(f"Deploy high-priority specialists ({', '.join(missing[:3])})", 7, "clinical", "short-term")
    elif gap >= 2:
        add(f"Recruit targeted specialists ({', '.join(missing)})", 5, "clinical", "mid-term")

    if doctor_gap > 0:
        needed_docs = int(doctor_gap * population_est / 1000)
        add(f"Workforce deficit: recruit ~{needed_docs} additional doctors (gap={doctor_gap:.2f}/1000)",
            8 if doctor_gap > 0.5 else 5, "workforce", "mid-term")

    if bed_gap > 0:
        needed_beds = int(bed_gap * population_est / 1000)
        add(f"Infrastructure deficit: add ~{needed_beds} beds (gap={bed_gap:.2f}/1000)",
            8 if bed_gap > 0.5 else 5, "infrastructure", "mid-term")

    if clinical_hospitals > 0 and beds_per_hospital < 10:
        add("Hospitals under-capacity → expand inpatient wards", 6, "infrastructure", "short-term")

    if clinical_hospitals > 0:
        if icu == 0:
            add("No ICU capability → upgrade at least one hospital to critical care", 9, "infrastructure", "immediate")
        elif icu / clinical_hospitals < 0.2:
            add("Insufficient ICU coverage (<20%) → expand critical care units", 7, "infrastructure", "mid-term")

    if em == 0:
        add("No emergency units → establish 24/7 emergency services", 9, "infrastructure", "immediate")
    elif total > 0 and em / total < 0.2:
        add("Low emergency coverage → expand emergency response network", 6, "infrastructure", "mid-term")

    if comp < 0.25:
        add("Critical data failure → conduct full audit + rebuild reporting pipeline", 8, "governance", "immediate")
    elif comp < 0.4:
        add("Improve facility reporting completeness", 5, "governance", "short-term")

    if anom > 10:
        add("High anomaly density → investigate fraud / misreporting risk", 8, "governance", "immediate")
    elif anom > 5:
        add("Moderate anomalies → validate facility data", 5, "governance", "short-term")

    if total <= 2:
        add("Extremely low facility density → deploy mobile clinics / outreach units", 9, "access", "immediate")
    elif total < 5:
        add("Low facility density → expand primary healthcare centers", 6, "access", "mid-term")

    if comp < 0.3 and anom > 5:
        add("Data unreliable → planning decisions may be invalid", 7, "governance", "immediate")

    risk_score = (gap * 1.5 + (1 if icu == 0 else 0) * 3 + (1 if em == 0 else 0) * 3
                  + (1 if docs < 10 else 0) * 2 + (1 if comp < 0.3 else 0) * 2 + (anom / 5))
    if risk_score > 10:
        add("HIGH RISK: Region likely to fail under demand surge", 9, "system", "immediate")

    if not priority_actions:
        return ["Monitor — no major intervention required"]

    priority_actions = sorted(priority_actions, key=lambda x: x["severity"], reverse=True)
    return [f"[{a['domain'].upper()} | {a['horizon']}] {a['action']}" for a in priority_actions]

plan_udf = F.udf(
    lambda gap, icu, em, docs, comp, anom, total, hospitals, beds, missing, clin_hosp: _build_plan({
        "critical_specialty_gap_count": gap,
        "icu_facilities":               icu,
        "emergency_medicine_facilities":em,
        "total_doctors":                docs,
        "avg_completeness":             comp,
        "total_region_anomalies":       anom,
        "total_facilities":             total,
        "hospital_count":               hospitals,
        "total_beds":                   beds,
        "missing_critical_specialties": missing,
        "clinical_hospital_count":      clin_hosp,
    }),
    ArrayType(StringType())
)

regional = regional.withColumn(
    "recommended_actions",
    plan_udf(
        F.col("critical_specialty_gap_count"),
        F.col("icu_facilities"),
        F.col("emergency_medicine_facilities"),
        F.col("total_doctors"),
        F.col("avg_completeness"),
        F.col("total_region_anomalies"),
        F.col("total_facilities"),
        F.col("hospital_count"),
        F.col("total_beds"),
        F.col("missing_critical_specialties"),
        F.col("clinical_hospital_count"),
    )
)

# COMMAND ----------
# MAGIC %md ## 11. Medical Desert Score

# COMMAND ----------

GHANA_REGION_POPULATION = {
    "Greater Accra": 5_401_964, "Ashanti": 5_780_438,
    "Western":       2_376_021, "Central": 2_563_228,
    "Eastern":       2_633_154, "Volta":   1_693_143,
    "Northern":      2_479_461, "Brong-Ahafo": 1_724_847,
    "Upper East":    1_046_545, "Upper West": 901_683,
    "Oti":             663_338, "Bono East": 1_141_427,
    "Ahafo":           564_397, "Savannah":  606_986,
    "North East":      527_170, "Western North": 782_022,
    "Unknown":       1_000_000,
}
WHO_MIN_FACILITIES_PER_100K = 5.0
WHO_MIN_BEDS_PER_10K        = 10.0
WHO_MIN_DOCTORS_PER_10K     = 1.0

def _compute_desert_score(region, total_fac, hospitals, total_docs, total_beds,
                           gap_count, icu_fac, em_fac, avg_comp):
    # Unknown region → sentinel (data insufficient to score)
    if not region or str(region).strip() in ("Unknown", "", "None"):
        return 0.5

    pop = GHANA_REGION_POPULATION.get(str(region or "Unknown"), 1_000_000)

    # Component 1: facility density (30%) — use all facilities
    fac_per_100k = (float(total_fac or 0) / pop) * 100_000
    density = max(0.0, 1.0 - min(1.0, fac_per_100k / WHO_MIN_FACILITIES_PER_100K))

    # Component 2: specialist gap (25%)
    specialist = min(1.0, float(gap_count or 0) / 8.0)

    # Component 3: infrastructure (20%) — data-availability discount for null-dominated fields
    beds_per_10k = (float(total_beds or 0) / pop) * 10_000
    docs_per_10k = (float(total_docs or 0) / pop) * 10_000
    has_bed_data = float(total_beds or 0) > 0
    has_doc_data = float(total_docs or 0) > 0

    if has_bed_data or has_doc_data:
        bed_sub = min(1.0, beds_per_10k / WHO_MIN_BEDS_PER_10K) if has_bed_data else 0.0
        doc_sub = min(1.0, docs_per_10k / WHO_MIN_DOCTORS_PER_10K) if has_doc_data else 0.0
        if has_bed_data and has_doc_data:
            infra = max(0.0, 1.0 - (0.6 * bed_sub + 0.4 * doc_sub))
        elif has_bed_data:
            infra = max(0.0, 1.0 - bed_sub) * 0.8  # 20% data-gap discount
        else:
            infra = max(0.0, 1.0 - doc_sub) * 0.8
    else:
        infra = 0.6  # data-availability penalty (not worst-case)

    # Component 4: completeness gap (15%)
    comp_gap = max(0.0, 1.0 - float(avg_comp or 0.0))

    # Component 5: emergency/ICU isolation (10%)
    isolation = 1.0 if (float(icu_fac or 0) == 0 and float(hospitals or 0) > 0) else \
                0.5 if float(em_fac or 0) == 0 else 0.0

    score = (0.30 * density + 0.25 * specialist + 0.20 * infra
             + 0.15 * comp_gap + 0.10 * isolation)
    return float(round(min(1.0, max(0.0, score)), 4))

desert_score_udf = F.udf(
    lambda reg, tf, ho, td, tb, gc, iu, em, ac:
        _compute_desert_score(reg, tf, ho, td, tb, gc, iu, em, ac),
    FloatType()
)

regional = regional.withColumn(
    "medical_desert_score",
    desert_score_udf(
        F.col("region_normalised"),
        F.col("total_facilities"),
        F.col("hospital_count"),
        F.col("total_doctors"),
        F.col("total_beds"),
        F.col("critical_specialty_gap_count"),
        F.col("icu_facilities"),
        F.col("emergency_medicine_facilities"),
        F.col("avg_completeness"),
    )
).withColumn(
    "desert_label",
    F.when(F.col("region_normalised") == "Unknown", "Data Insufficient")
     .when(F.col("medical_desert_score") > 0.75, "Critical Desert")
     .when(F.col("medical_desert_score") > 0.55, "Severe Desert")
     .when(F.col("medical_desert_score") > 0.35, "Moderate Desert")
     .when(F.col("medical_desert_score") > 0.15, "At Risk")
     .otherwise("Adequately Served")
)

# Join desert score back to facility-level gold
gold = gold.join(
    regional.select("region_normalised", "medical_desert_score", "desert_label"),
    on="region_normalised",
    how="left"
)

print("✅ Desert scores computed.")

# COMMAND ----------
# MAGIC %md ## 12. Write Gold Tables (with Canonical Schema GOLD-V5-1)

# COMMAND ----------

def _normalize_phone(phone):
    if phone is None:
        return None
    s = str(phone).strip()
    if s.lower() in ("", "null", "none", "nan"):
        return None
    if s.startswith("+"):
        digits = "+" + re.sub(r"\D", "", s[1:])
        return digits if len(digits) > 1 else None
    digits = re.sub(r"\D", "", s)
    if len(digits) == 10 and digits.startswith("0"):
        return "+233" + digits[1:]
    if len(digits) == 9:
        return "+233" + digits
    if len(digits) == 12 and digits.startswith("233"):
        return "+" + digits
    return None

def _normalize_phone_list(raw):
    if raw is None:
        return []
    items = raw if isinstance(raw, list) else (
        json.loads(raw) if str(raw).strip() not in ("", "null", "None", "nan") else []
    )
    out = []
    for p in items:
        n = _normalize_phone(p)
        if n:
            out.append(n)
    return list(dict.fromkeys(out))

normalize_phone_list_udf = F.udf(_normalize_phone_list, ArrayType(StringType()))
country_udf = F.udf(
    lambda c: "Ghana" if str(c or "").strip().lower() in ("gh", "ghana") else str(c or "").strip().title() or None,
    StringType()
)
country_code_udf = F.udf(
    lambda country, code: (str(code).strip().upper() if code and str(code).strip() else None)
                          or ("GH" if str(country or "").strip() == "Ghana" else None),
    StringType()
)

INTERNAL_COLS = ["_row_richness", "_name_norm", "_city_norm", "_r1", "_r2"]
gold_clean = gold.drop(*[c for c in INTERNAL_COLS if c in gold.columns])

gold_clean = (
    gold_clean
    .withColumn("phone_numbers",    normalize_phone_list_udf(F.col("phone_numbers")))
    .withColumn("official_phone",
        F.when(F.size(F.col("phone_numbers")) > 0,
               F.element_at(F.col("phone_numbers"), 1).cast(StringType())))
    .withColumn("address_country",
        country_udf(F.col("address_country")))
    .withColumn("address_countrycode",
        country_code_udf(F.col("address_country"), F.col("address_countrycode")))
)

# GOLD-V5-1: Canonical schema aliases matching exact PDF field names
gold_clean = gold_clean \
    .withColumn("officialWebsite",       F.col("officialWebsite")) \
    .withColumn("address_stateOrRegion", F.col("address_stateorregion")) \
    .withColumn("address_zipOrPostcode", F.col("address_ziporpostcode")) \
    .withColumn("address_countryCode",   F.col("address_countrycode")) \
    .withColumn("facilityTypeId",        F.col("facility_type_clean")) \
    .withColumn("operatorTypeId",
        F.when(F.col("operatortypeid").isNotNull(), F.lower(F.col("operatortypeid")))
         .otherwise(F.lit(None).cast(StringType()))) \
    .withColumn("numberDoctors",         F.col("number_doctors_int")) \
    .withColumn("extraction_version",    F.lit("gold_v5"))

print("✅ Canonical schema aliases added.")

# Write gold facilities
(
    gold_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .saveAsTable(cfg.GOLD_FACILITIES)
)
print(f"✅ {cfg.GOLD_FACILITIES}  rows: {gold_clean.count():,}")

# Write regional summary
(
    regional.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(cfg.GOLD_REGIONAL)
)
print(f"✅ {cfg.GOLD_REGIONAL}  rows: {regional.count():,}")

# COMMAND ----------
# MAGIC %md ## 13. Gold Quality Report

# COMMAND ----------

g  = spark.table(cfg.GOLD_FACILITIES)
gr = spark.table(cfg.GOLD_REGIONAL)
total = g.count()

print("=" * 65)
print(f"GOLD FACILITIES — Key Metrics ({total:,} rows)")
print("=" * 65)

flag_cols = [
    "is_hospital", "is_clinic", "is_ngo",
    "has_emergency_medicine", "has_obstetrics", "has_surgery", "has_icu",
    "has_radiology", "has_infectious_disease", "has_mental_health",
    "stat_anomaly_capability_inflation", "stat_anomaly_hospital_no_doctors",
    "stat_anomaly_ghost_facility", "stat_anomaly_clinic_claims_icu",
    "stat_anomaly_specialty_mismatch", "stat_anomaly_procedure_breadth",
    "is_rag_ready",
]
for col_name in flag_cols:
    if col_name not in g.columns:
        print(f"  {'MISSING':8} {col_name}")
        continue
    n   = g.filter(F.col(col_name) == True).count()
    pct = n / total * 100
    print(f"  {col_name:<50} {n:>5,}  ({pct:4.1f}%)")

print()
print("=" * 65)
print(f"GOLD REGIONAL SUMMARY — {gr.count()} regions")
print("=" * 65)

display(
    gr.select(
        "region_normalised", "total_facilities", "clinical_facility_count",
        "hospital_count", "ngo_count",
        "total_doctors", "total_beds", "emergency_medicine_facilities",
        "icu_facilities", "critical_specialty_gap_count",
        "missing_critical_specialties",
        "avg_completeness", "medical_desert_score", "desert_label",
        "procedure_breadth_anomalies",
    ).orderBy(F.desc("medical_desert_score"))
)

non_canonical = gr.filter(
    ~F.col("region_normalised").isin(list(CANONICAL_REGIONS) + ["Unknown"])
).select("region_normalised", "total_facilities")
print(f"\nNon-canonical regions remaining: {non_canonical.count()}")
if non_canonical.count() > 0:
    non_canonical.show()

# COMMAND ----------
# MAGIC %md ## 14. MLflow Tracing (GOLD-V5-10)

# COMMAND ----------

try:
    import mlflow

    mlflow.set_experiment("/virtue-foundation/pipeline")

    with mlflow.start_run(run_name="03_gold_aggregate_v5") as run:
        # Core counts
        mlflow.log_metric("total_facilities",    total)
        mlflow.log_metric("hospital_count",      g.filter(F.col("is_hospital")).count())
        mlflow.log_metric("ngo_count",           g.filter(F.col("is_ngo")).count())
        mlflow.log_metric("rag_ready_count",     g.filter(F.col("is_rag_ready")).count())
        mlflow.log_metric("total_anomaly_flags", g.agg(F.sum("total_stat_anomalies")).collect()[0][0] or 0)
        mlflow.log_metric("regions_canonical",   gr.filter(F.col("region_normalised") != "Unknown").count())
        mlflow.log_metric("avg_completeness",    g.agg(F.avg("data_completeness_score")).collect()[0][0] or 0)

        # Agentic step tracing (GOLD-V5-10): log each reasoning stage
        # Step 1: Region consolidation
        mlflow.log_param("step_1_region_consolidation", "unknown_rate=" + str(round(unknown_ct/total_ct*100,1)) + "%")
        # Step 2: Clinical feature extraction
        mlflow.log_metric("step_2_has_emergency_medicine", g.filter(F.col("has_emergency_medicine")).count())
        mlflow.log_metric("step_2_has_surgery",            g.filter(F.col("has_surgery")).count())
        mlflow.log_metric("step_2_has_obstetrics",         g.filter(F.col("has_obstetrics")).count())
        mlflow.log_metric("step_2_has_icu",                g.filter(F.col("has_icu")).count())
        # Step 3: Anomaly detection
        mlflow.log_metric("step_3_anomaly_capability_inflation", g.filter(F.col("stat_anomaly_capability_inflation")).count())
        mlflow.log_metric("step_3_anomaly_ghost_facility",       g.filter(F.col("stat_anomaly_ghost_facility")).count())
        mlflow.log_metric("step_3_anomaly_procedure_breadth",    g.filter(F.col("stat_anomaly_procedure_breadth")).count())
        # Step 4: Desert scoring
        desert_counts = gr.groupBy("desert_label").count().collect()
        for row in desert_counts:
            label_key = re.sub(r'[^a-z0-9]', '_', str(row["desert_label"] or "").lower())
            mlflow.log_metric(f"step_4_desert_{label_key}", row["count"])
        # Params
        mlflow.log_param("gold_facilities_table", cfg.GOLD_FACILITIES)
        mlflow.log_param("gold_regional_table",   cfg.GOLD_REGIONAL)
        mlflow.log_param("extraction_version",    "gold_v5")
        mlflow.log_param("canonical_schema",      "PDF_compliant_v5")

        print(f"MLflow run logged: {run.info.run_id} ✅")

except Exception as e:
    print(f"MLflow skipped: {e}")

# COMMAND ----------
# MAGIC %sql
# MAGIC SELECT
# MAGIC   unique_id, name, region_normalised, facilityTypeId, operatorTypeId,
# MAGIC   officialWebsite, address_stateOrRegion, address_countryCode,
# MAGIC   numberDoctors, capacity_int, medical_desert_score, desert_label,
# MAGIC   has_emergency_medicine, has_surgery, has_icu,
# MAGIC   total_stat_anomalies, is_rag_ready
# MAGIC FROM virtue_foundation.ghana.gold_facilities_enriched
# MAGIC ORDER BY medical_desert_score DESC
# MAGIC LIMIT 50

# COMMAND ----------
# MAGIC %sql
# MAGIC SELECT * FROM virtue_foundation.ghana.gold_regional_summary
# MAGIC ORDER BY medical_desert_score DESC

#######################################
# OUTPUT OF THE NOTEBOOK
######################################

'''
Run: 2026-04-25T13:52:35.330557+00:00
Silver rows    : 909
Silver columns : 90
Region distribution after consolidation:
+-----------------+-----+
|region_normalised|count|
+-----------------+-----+
|    Greater Accra|  409|
|          Ashanti|  159|
|          Western|   78|
|          Unknown|   40|
|          Central|   38|
|            Volta|   36|
|         Northern|   31|
|      Brong-Ahafo|   29|
|          Eastern|   26|
|       Upper West|   13|
|    Western North|   12|
|        Bono East|   11|
|              Oti|    7|
|       Upper East|    6|
|            Ahafo|    6|
|         Savannah|    5|
|       North East|    3|
+-----------------+-----+

Unknown region: 40 / 909 (4.4%)
✅ Facility-level feature engineering complete.
✅ Anomaly flags complete.
RAG-ready rows: 772 / 909
✅ Regional aggregation complete.
✅ Desert scores computed.
✅ Canonical schema aliases added.
✅ virtue_foundation.ghana.gold_facilities_enriched  rows: 909
✅ virtue_foundation.ghana.gold_regional_summary  rows: 17
=================================================================
GOLD FACILITIES — Key Metrics (909 rows)
=================================================================
  is_hospital                                          447  (49.2%)
  is_clinic                                            369  (40.6%)
  is_ngo                                                65  ( 7.2%)
  has_emergency_medicine                                69  ( 7.6%)
  has_obstetrics                                       189  (20.8%)
  has_surgery                                          107  (11.8%)
  has_icu                                               14  ( 1.5%)
  has_radiology                                        100  (11.0%)
  has_infectious_disease                                79  ( 8.7%)
  has_mental_health                                     50  ( 5.5%)
  stat_anomaly_capability_inflation                    128  (14.1%)
  stat_anomaly_hospital_no_doctors                      78  ( 8.6%)
  stat_anomaly_ghost_facility                            0  ( 0.0%)
  stat_anomaly_clinic_claims_icu                         1  ( 0.1%)
  stat_anomaly_specialty_mismatch                       11  ( 1.2%)
  stat_anomaly_procedure_breadth                         6  ( 0.7%)
  is_rag_ready                                         772  (84.9%)

=================================================================
GOLD REGIONAL SUMMARY — 17 regions
=================================================================
'''

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT * FROM virtue_foundation.ghana.gold_facilities_enriched

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT * FROM virtue_foundation.ghana.gold_regional_summary

In [0]:
%sql
SELECT * FROM virtue_foundation.ghana.gold_facilities_enriched LIMIT 200

In [0]:
%sql
SELECT * FROM virtue_foundation.ghana.gold_regional_summary

In [0]:
%sql
SHOW CREATE TABLE virtue_foundation.ghana.gold_regional_summary

In [0]:
%sql
SHOW CREATE TABLE virtue_foundation.ghana.gold_facilities_enriched